In [0]:
# BLOCO 0
# Este bloco importa todas as bibliotecas necessárias para o pipeline Medallion
# (Bronze → Silver → Gold) no Databricks.

# ================================
# BIBLIOTECAS BASE
# ================================

import pandas as pd
import numpy as np

# ================================
# PYSPARK
# ================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ================================
# TRATAMENTO DE COLUNAS
# ================================

import unicodedata
import re

# ================================
# CONFIGURAÇÕES (OPCIONAL)
# ================================

# Melhor visualização de dados no pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# ================================
# FUNÇÃO AUXILIAR (IMPORTANTE PARA SILVER)
# ================================
# Remove acentos, espaços e caracteres especiais dos nomes de colunas

def normalizar_nome_coluna(nome):
    nome = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')
    nome = nome.replace(" ", "_")
    nome = re.sub(r'[^a-zA-Z0-9_]', '', nome)
    nome = nome.lower()
    return nome

print("Ambiente configurado com sucesso.")

In [0]:
# BLOCO 1
# Este bloco configura o catálogo/schema de trabalho e verifica se os arquivos
# necessários estão disponíveis no Volume do Databricks antes de iniciar a camada Bronze.

# Define o catálogo e o schema do projeto
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("USE workspace.default")

# Caminho base dos arquivos no Volume
caminho_base = "/Volumes/workspace/default/next_cargas/"

# Lista os arquivos disponíveis no Volume
arquivos = dbutils.fs.ls(caminho_base)

print("Arquivos encontrados no Volume:")
for arquivo in arquivos:
    print(arquivo.name)

# **CAMADA BRONZE (Dados Brutos)**

In [0]:
# BLOCO 2
# Este bloco lê o arquivo Cadastros.xlsx com pandas, converte para DataFrame Spark
# e grava os dados brutos na camada Bronze em formato Delta, sem transformações.

# Caminho do arquivo de origem
caminho_cadastros = "/Volumes/workspace/default/next_cargas/Cadastros.xlsx"

# Nome da tabela Bronze
tabela_bronze_cadastros = "workspace.default.bronze_cadastros"

# Leitura do Excel com pandas
df_cadastros_pd = pd.read_excel(caminho_cadastros)

# Converte todas as colunas para string na Bronze, preservando o dado bruto
df_cadastros_pd = df_cadastros_pd.astype(str)

# Converte para Spark DataFrame
df_cadastros_bronze = spark.createDataFrame(df_cadastros_pd)

# Gravação na camada Bronze em Delta
(
    df_cadastros_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_bronze_cadastros)
)

# Visualização inicial
display(df_cadastros_bronze)

# Conferência da tabela gravada
display(spark.table(tabela_bronze_cadastros))

In [0]:
# BLOCO 3
# Este bloco lê o arquivo fFrete.csv com encoding UTF-16 e separador tab,
# preserva os dados brutos sem transformações e grava na camada Bronze em Delta.

# Caminho do arquivo de origem
caminho_frete = "/Volumes/workspace/default/next_cargas/fFrete.csv"

# Nome da tabela Bronze
tabela_bronze_frete = "workspace.default.bronze_frete"

# Leitura do CSV bruto
df_frete_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", "\t")
    .option("encoding", "UTF-16")
    .option("inferSchema", "false")
    .load(caminho_frete)
)

# Gravação na camada Bronze em Delta
(
    df_frete_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_bronze_frete)
)

# Visualização inicial
display(df_frete_bronze)

# Conferência da tabela gravada
display(spark.table(tabela_bronze_frete))

In [0]:
# BLOCO 4
# Este bloco lê o arquivo fKMRodado.xlsx com pandas, converte para DataFrame Spark
# e grava os dados brutos na camada Bronze em formato Delta, sem transformações.

import pandas as pd

# Caminho do arquivo de origem
caminho_km_rodado = "/Volumes/workspace/default/next_cargas/fKMRodado.xlsx"

# Nome da tabela Bronze
tabela_bronze_km_rodado = "workspace.default.bronze_km_rodado"

# Leitura do Excel com pandas
df_km_rodado_pd = pd.read_excel(caminho_km_rodado)

# Converte todas as colunas para string na Bronze, preservando o dado bruto
df_km_rodado_pd = df_km_rodado_pd.astype(str)

# Converte para Spark DataFrame
df_km_rodado_bronze = spark.createDataFrame(df_km_rodado_pd)

# Gravação na camada Bronze em Delta
(
    df_km_rodado_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_bronze_km_rodado)
)

# Visualização inicial
display(df_km_rodado_bronze)

# Conferência da tabela gravada
display(spark.table(tabela_bronze_km_rodado))

# **CAMADA SILVER (Tratamento e Padronização)**

In [0]:
# BLOCO 5
# Este bloco transforma a tabela Bronze de Cadastros em Silver,
# normalizando nomes de colunas, tratando nulos e ajustando tipos de dados.

from pyspark.sql.functions import col, trim, when

# Origem Bronze e destino Silver
tabela_bronze_cadastros = "workspace.default.bronze_cadastros"
tabela_silver_cadastros = "workspace.default.silver_cadastros"

# Leitura da Bronze
df_cadastros_silver = spark.table(tabela_bronze_cadastros)

# Normalização dos nomes das colunas
novos_nomes = [normalizar_nome_coluna(c) for c in df_cadastros_silver.columns]
df_cadastros_silver = df_cadastros_silver.toDF(*novos_nomes)

# Limpeza e tipagem
df_cadastros_silver = (
    df_cadastros_silver
    .withColumn("sk_motorista", col("sk_motorista").cast("int"))
    .withColumn("motorista", trim(col("motorista")))
    .withColumn("motorista", when(col("motorista") == "nan", None).otherwise(col("motorista")))
)

# Tratamento de nulos essenciais
df_cadastros_silver = df_cadastros_silver.dropna(subset=["sk_motorista", "motorista"])

# Remoção de duplicados, se houver
df_cadastros_silver = df_cadastros_silver.dropDuplicates(["sk_motorista"])

# Gravação da Silver
(
    df_cadastros_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver_cadastros)
)

# Visualização
display(df_cadastros_silver)

# Conferência final
display(spark.table(tabela_silver_cadastros))

In [0]:
# BLOCO 6
# Este bloco transforma a tabela Bronze de Frete em Silver,
# normalizando nomes de colunas, ajustando tipos de dados
# e tratando nulos para preparar a base analítica.

from pyspark.sql.functions import col, trim, when, regexp_replace, to_date

# Origem Bronze e destino Silver
tabela_bronze_frete = "workspace.default.bronze_frete"
tabela_silver_frete = "workspace.default.silver_frete"

# Leitura da Bronze
df_frete_silver = spark.table(tabela_bronze_frete)

# Normalização dos nomes das colunas
novos_nomes = [normalizar_nome_coluna(c) for c in df_frete_silver.columns]
df_frete_silver = df_frete_silver.toDF(*novos_nomes)

# Ajuste inicial de espaços e valores textuais inválidos
for coluna in df_frete_silver.columns:
    df_frete_silver = df_frete_silver.withColumn(coluna, trim(col(coluna)))

# Trata strings "nan", "null" e vazias como nulo
for coluna in df_frete_silver.columns:
    df_frete_silver = df_frete_silver.withColumn(
        coluna,
        when(col(coluna).isin("nan", "null", ""), None).otherwise(col(coluna))
    )

# Conversão de data
# Ajuste o formato caso sua origem esteja em outro padrão
df_frete_silver = df_frete_silver.withColumn(
    "data",
    to_date(col("data"), "dd/MM/yyyy")
)

# Função auxiliar para converter números que podem vir com ponto de milhar e vírgula decimal
def converter_decimal(coluna):
    return regexp_replace(
        regexp_replace(col(coluna), r"\.", ""),
        ",",
        "."
    ).cast("double")

# Conversão dos campos numéricos
df_frete_silver = (
    df_frete_silver
    .withColumn("sk_cliente", col("sk_cliente").cast("int"))
    .withColumn("sk_veiculo", col("sk_veiculo").cast("int"))
    .withColumn("numero_documento_fiscal", col("numero_documento_fiscal").cast("string"))
    .withColumn("viagem", col("viagem").cast("string"))
    .withColumn("cod_ibge", col("cod_ibge").cast("string"))
    .withColumn("valor_do_frete_liquido", converter_decimal("valor_do_frete_liquido"))
    .withColumn("peso_kg", converter_decimal("peso_kg"))
    .withColumn("peso_cubado", converter_decimal("peso_cubado"))
    .withColumn("valor_da_mercadoria", converter_decimal("valor_da_mercadoria"))
)

# Tratamento de nulos em métricas numéricas
df_frete_silver = (
    df_frete_silver
    .fillna({"valor_do_frete_liquido": 0.0})
    .fillna({"peso_kg": 0.0})
    .fillna({"peso_cubado": 0.0})
    .fillna({"valor_da_mercadoria": 0.0})
)

# Remove registros sem data ou sem chave mínima de veículo
df_frete_silver = df_frete_silver.dropna(subset=["data", "sk_veiculo"])

# Remove duplicados com base em um conjunto mínimo de negócio
df_frete_silver = df_frete_silver.dropDuplicates([
    "data",
    "sk_veiculo",
    "numero_documento_fiscal",
    "viagem"
])

# Gravação da Silver
(
    df_frete_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver_frete)
)

# Visualização
display(df_frete_silver)

# Conferência final
display(spark.table(tabela_silver_frete))

In [0]:
# BLOCO 7
# Este bloco transforma a tabela Bronze de KM Rodado em Silver,
# normalizando nomes de colunas, ajustando tipos de dados
# e tratando diferentes formatos da coluna mes.

from pyspark.sql.functions import col, trim, when, regexp_replace, to_date, concat, lit, length

# Origem Bronze e destino Silver
tabela_bronze_km_rodado = "workspace.default.bronze_km_rodado"
tabela_silver_km_rodado = "workspace.default.silver_km_rodado"

# Leitura da Bronze
df_km_silver = spark.table(tabela_bronze_km_rodado)

# Normalização dos nomes das colunas
novos_nomes = [normalizar_nome_coluna(c) for c in df_km_silver.columns]
df_km_silver = df_km_silver.toDF(*novos_nomes)

# Ajuste inicial de espaços
for coluna in df_km_silver.columns:
    df_km_silver = df_km_silver.withColumn(coluna, trim(col(coluna)))

# Trata strings inválidas como nulo
for coluna in df_km_silver.columns:
    df_km_silver = df_km_silver.withColumn(
        coluna,
        when(col(coluna).isin("nan", "null", ""), None).otherwise(col(coluna))
    )

# Função auxiliar para converter números com ponto de milhar e vírgula decimal
def converter_decimal(coluna):
    return regexp_replace(
        regexp_replace(col(coluna), r"\.", ""),
        ",",
        "."
    ).cast("double")

# Conversão robusta do campo mes
# Suporta:
# - MM/yyyy  -> ex: 01/2018
# - yyyy-MM-dd -> ex: 2018-01-01
df_km_silver = df_km_silver.withColumn(
    "mes",
    when(length(col("mes")) == 7, to_date(concat(lit("01/"), col("mes")), "dd/MM/yyyy"))
    .when(length(col("mes")) == 10, to_date(col("mes"), "yyyy-MM-dd"))
    .otherwise(None)
)

# Conversão dos campos numéricos
df_km_silver = (
    df_km_silver
    .withColumn("sk_veiculo", col("sk_veiculo").cast("int"))
    .withColumn("sk_motorista", col("sk_motorista").cast("int"))
    .withColumn("km_percorrido", converter_decimal("km_percorrido"))
    .withColumn("litros", converter_decimal("litros"))
    .withColumn("combustivel", converter_decimal("combustivel"))
    .withColumn("manutencao", converter_decimal("manutencao"))
    .withColumn("custo_fixo", converter_decimal("custo_fixo"))
)

# Tratamento de nulos nas métricas numéricas
df_km_silver = (
    df_km_silver
    .fillna({"km_percorrido": 0.0})
    .fillna({"litros": 0.0})
    .fillna({"combustivel": 0.0})
    .fillna({"manutencao": 0.0})
    .fillna({"custo_fixo": 0.0})
)

# Remove registros sem mês, veículo ou motorista
df_km_silver = df_km_silver.dropna(subset=["mes", "sk_veiculo", "sk_motorista"])

# Remove duplicidades por chave de negócio mensal
df_km_silver = df_km_silver.dropDuplicates(["mes", "sk_veiculo", "sk_motorista"])

# Gravação da Silver
(
    df_km_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver_km_rodado)
)

# Visualização
display(df_km_silver)

# Conferência final
display(spark.table(tabela_silver_km_rodado))

# CAMADA GOLD (Agregações e KPIs)

In [0]:
# BLOCO 8
# Este bloco cria a primeira tabela Gold com KPIs de frete por mês,
# agregando a base Silver de frete para análise de negócio.

from pyspark.sql.functions import year, month, countDistinct, sum, round, col

# Origem Silver e destino Gold
tabela_silver_frete = "workspace.default.silver_frete"
tabela_gold_kpi_frete_mes = "workspace.default.gold_kpi_frete_mes"

# Leitura da Silver
df_frete_gold = spark.table(tabela_silver_frete)

# Criação da Gold com agregações mensais
df_gold_kpi_frete_mes = (
    df_frete_gold
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
    .groupBy("ano", "mes")
    .agg(
        countDistinct("viagem").alias("quantidade_viagens"),
        countDistinct("numero_documento_fiscal").alias("quantidade_documentos_fiscais"),
        round(sum("valor_do_frete_liquido"), 2).alias("valor_total_frete_liquido"),
        round(sum("peso_kg"), 2).alias("peso_total_kg"),
        round(sum("peso_cubado"), 2).alias("peso_total_cubado"),
        round(sum("valor_da_mercadoria"), 2).alias("valor_total_mercadoria")
    )
    .withColumn(
        "ticket_medio_por_viagem",
        round(col("valor_total_frete_liquido") / col("quantidade_viagens"), 2)
    )
    .orderBy("ano", "mes")
)

# Gravação da Gold
(
    df_gold_kpi_frete_mes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_kpi_frete_mes)
)

# Visualização
display(df_gold_kpi_frete_mes)

# Conferência final
display(spark.table(tabela_gold_kpi_frete_mes))

In [0]:
# BLOCO 9
# Este bloco cria uma tabela Gold de custo por motorista por mês,
# cruzando a base de KM Rodado com o cadastro de motoristas.

from pyspark.sql.functions import year, month, round, sum, col, when

# Origem Silver e destino Gold
tabela_silver_km_rodado = "workspace.default.silver_km_rodado"
tabela_silver_cadastros = "workspace.default.silver_cadastros"
tabela_gold_custo_motorista_mes = "workspace.default.gold_custo_motorista_mes"

# Leitura das tabelas Silver
df_km_gold = spark.table(tabela_silver_km_rodado)
df_cadastros_gold = spark.table(tabela_silver_cadastros)

# Join com cadastro de motoristas
df_base_custo_motorista = (
    df_km_gold.alias("km")
    .join(
        df_cadastros_gold.alias("cad"),
        col("km.sk_motorista") == col("cad.sk_motorista"),
        "left"
    )
    .select(
        col("km.mes"),
        col("km.sk_motorista"),
        col("cad.motorista"),
        col("km.sk_veiculo"),
        col("km.km_percorrido"),
        col("km.litros"),
        col("km.combustivel"),
        col("km.manutencao"),
        col("km.custo_fixo")
    )
)

# Agregação por ano, mês e motorista
df_gold_custo_motorista_mes = (
    df_base_custo_motorista
    .withColumn("ano", year(col("mes")))
    .withColumn("numero_mes", month(col("mes")))
    .groupBy("ano", "numero_mes", "sk_motorista", "motorista")
    .agg(
        round(sum("km_percorrido"), 2).alias("km_total"),
        round(sum("litros"), 2).alias("litros_total"),
        round(sum("combustivel"), 2).alias("custo_total_combustivel"),
        round(sum("manutencao"), 2).alias("custo_total_manutencao"),
        round(sum("custo_fixo"), 2).alias("custo_total_fixo")
    )
    .withColumn(
        "custo_total_operacional",
        round(
            col("custo_total_combustivel") +
            col("custo_total_manutencao") +
            col("custo_total_fixo"),
            2
        )
    )
    .withColumn(
        "custo_por_km",
        when(col("km_total") > 0,
             round(col("custo_total_operacional") / col("km_total"), 4))
        .otherwise(0.0)
    )
    .withColumn(
        "custo_medio_por_litro",
        when(col("litros_total") > 0,
             round(col("custo_total_combustivel") / col("litros_total"), 4))
        .otherwise(0.0)
    )
    .orderBy("ano", "numero_mes", "motorista")
)

# Gravação da Gold
(
    df_gold_custo_motorista_mes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_custo_motorista_mes)
)

# Visualização
display(df_gold_custo_motorista_mes)

# Conferência final
display(spark.table(tabela_gold_custo_motorista_mes))

In [0]:
# BLOCO 10
# Este bloco cria uma tabela Gold com indicadores mensais de KM rodado da frota,
# consolidando volume operacional, consumo e custos.

from pyspark.sql.functions import year, month, round, sum, countDistinct, col, when

# Origem Silver e destino Gold
tabela_silver_km_rodado = "workspace.default.silver_km_rodado"
tabela_gold_km_rodado_mes = "workspace.default.gold_km_rodado_mes"

# Leitura da Silver
df_km_gold = spark.table(tabela_silver_km_rodado)

# Criação da Gold com agregações mensais
df_gold_km_rodado_mes = (
    df_km_gold
    .withColumn("ano", year(col("mes")))
    .withColumn("numero_mes", month(col("mes")))
    .groupBy("ano", "numero_mes")
    .agg(
        round(sum("km_percorrido"), 2).alias("km_total"),
        round(sum("litros"), 2).alias("litros_total"),
        round(sum("combustivel"), 2).alias("custo_total_combustivel"),
        round(sum("manutencao"), 2).alias("custo_total_manutencao"),
        round(sum("custo_fixo"), 2).alias("custo_total_fixo"),
        countDistinct("sk_veiculo").alias("quantidade_veiculos_ativos"),
        countDistinct("sk_motorista").alias("quantidade_motoristas_ativos")
    )
    .withColumn(
        "custo_total_operacional",
        round(
            col("custo_total_combustivel") +
            col("custo_total_manutencao") +
            col("custo_total_fixo"),
            2
        )
    )
    .withColumn(
        "km_medio_por_veiculo",
        when(col("quantidade_veiculos_ativos") > 0,
             round(col("km_total") / col("quantidade_veiculos_ativos"), 2))
        .otherwise(0.0)
    )
    .withColumn(
        "consumo_medio_km_por_litro",
        when(col("litros_total") > 0,
             round(col("km_total") / col("litros_total"), 4))
        .otherwise(0.0)
    )
    .withColumn(
        "custo_por_km",
        when(col("km_total") > 0,
             round(col("custo_total_operacional") / col("km_total"), 4))
        .otherwise(0.0)
    )
    .orderBy("ano", "numero_mes")
)

# Gravação da Gold
(
    df_gold_km_rodado_mes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold_km_rodado_mes)
)

# Visualização
display(df_gold_km_rodado_mes)

# Conferência final
display(spark.table(tabela_gold_km_rodado_mes))

# CAMADA DE CONSUMO (BI)

In [0]:
# BLOCO 11
# Este bloco valida as tabelas da camada Gold, exibindo contagem de registros,
# schema e amostras dos dados para garantir que a modelagem final está pronta
# para consumo analítico no Power BI.

# Lista das tabelas Gold do projeto
tabelas_gold = [
    "workspace.default.gold_kpi_frete_mes",
    "workspace.default.gold_custo_motorista_mes",
    "workspace.default.gold_km_rodado_mes"
]

# Validação das tabelas Gold
for tabela in tabelas_gold:
    print("=" * 100)
    print(f"VALIDANDO TABELA: {tabela}")
    print("=" * 100)
    
    df = spark.table(tabela)
    
    print("Quantidade de registros:")
    print(df.count())
    
    print("\nSchema:")
    df.printSchema()
    
    print("\nAmostra dos dados:")
    display(df.limit(10))

In [0]:
%sql
-- BLOCO 12
-- Este bloco cria views finais para consumo no Power BI a partir da camada Gold.
-- A ideia é expor objetos estáveis, simples e prontos para análise.

CREATE OR REPLACE VIEW workspace.default.vw_bi_kpi_frete_mes AS
SELECT
    ano,
    mes,
    quantidade_viagens,
    quantidade_documentos_fiscais,
    valor_total_frete_liquido,
    peso_total_kg,
    peso_total_cubado,
    valor_total_mercadoria,
    ticket_medio_por_viagem
FROM workspace.default.gold_kpi_frete_mes;

CREATE OR REPLACE VIEW workspace.default.vw_bi_custo_motorista_mes AS
SELECT
    ano,
    numero_mes,
    sk_motorista,
    motorista,
    km_total,
    litros_total,
    custo_total_combustivel,
    custo_total_manutencao,
    custo_total_fixo,
    custo_total_operacional,
    custo_por_km,
    custo_medio_por_litro
FROM workspace.default.gold_custo_motorista_mes;

CREATE OR REPLACE VIEW workspace.default.vw_bi_km_rodado_mes AS
SELECT
    ano,
    numero_mes,
    km_total,
    litros_total,
    custo_total_combustivel,
    custo_total_manutencao,
    custo_total_fixo,
    custo_total_operacional,
    quantidade_veiculos_ativos,
    quantidade_motoristas_ativos,
    km_medio_por_veiculo,
    consumo_medio_km_por_litro,
    custo_por_km
FROM workspace.default.gold_km_rodado_mes;